
#Reversing Transcription and Translation using SMT solvers

Going from a DNA template strand to a protein is mechanical: transcribe to mRNA, then read
the mRNA three bases at a time. Going the other way is not, because most amino acids are
coded for by several different codons, so a protein does not determine its own mRNA. In
this notebook we will look at SMT solvers and see how Z3 handles that ambiguity, by
describing what a valid mRNA strand for a given protein looks like and asking the solver to
enumerate them.


**Instructions:**
1. To get started, click on File on the top left and click "Save a copy in Drive."
This will give you an editable version of this document that you can use.
2. If you press `CMD`+`Enter` it runs the cell, and if you press `Shift`+`Enter` it runs the cell and goes to the next one.
3. Make sure you run all cells as you go through the notebook; some cells will not work properly unless the previous one
has been run too.
4. If you disconnect or are inactive for some time you should run all of the cells again.

## 0. Preliminaries (you should run this cell but there is no need to read it)

In [ ]:
!pip install z3-solver
!pip install git+https://github.com/crrivero/FormalMethodsTasting.git#subdirectory=core
from z3 import *
from tofmcore import showSolver
from IPython.display import clear_output
clear_output()

## Encoding constraints in Z3

The goal of this notebook is to teach you about formal methods;
particularly, how you can use existing formal verification tools
(in this case, Z3) to analyze and solve your own problems.
Before we get started, let's look at some basic things we can do with Z3.

### Integers

Let's use Z3 to solve problems involving integers. Let's start with something simple: find $x$ such that

$$2x + 5 = 15$$

In [ ]:
# Initialize variables

x = Int('x') # declairing that x is an integer named 'x'

# Initialize Z3 solver
s = Solver()

s.add( 2*x + 5 == 15 ) # add the equation

print(s)
print(s.check())
print(s.model())

Now let's try to check whether that's the only solution. We can do this by adding the following constraint to the solver:

$$x \not= 5$$

If the solver returns "**unsat**" then $x=5$ is the only solution.
Try it yourself by completing the code in the cell below.

In [ ]:
s.add( x == 5 ) # REPLACE THIS LINE
s.check()

## DNA Transcription/Translation

Proteins are formed in the body first by transcription of DNA to mRNA, and then translation of mRNA to polymers, which are chains of amino acids. Let's see this first with some code:

First, we define a function for transcription: taking a DNA template strand and transcribing it into its corresponding mRNA strand. For example, if we gave it `TAC`, it would return `AUG`.

In [ ]:
def transcribe(dna: str) -> str:
    mapping = {
        'A': 'U',
        'T': 'A',
        'G': 'C',
        'C': 'G',
    }
    return ''.join(mapping[b] for b in dna)

We'll define another function for translation: taking an mRNA strand and translating it into its correspdoning polymer. For example, if we gave it `AUGUUA`, we would get `['Met', Leu']`.

In [ ]:
# mRNA codon table
# Maps codons to their corresponding amino acids
# Each line belonds to one amino acid
codon_to_aa = {
    'AUG': 'Met',  # start
    'UUU': 'Phe', 'UUC': 'Phe',
    'UUA': 'Leu', 'UUG': 'Leu',
    'UAA': 'Stop', 'UAG': 'Stop', 'UGA': 'Stop'
}


# mRNA -> protein translation
def translate(mrna: str):
    protein = []

    # Iterate through the mRNA strand in chunks of three
    for i in range(0, len(mrna), 3):
        codon = mrna[i:i+3]
        if len(codon) < 3:
            break

        aa = codon_to_aa.get(codon, '?')  # '?' for unlisted codons

        if aa == 'Stop':
            break

        protein.append(aa)

    return protein


Here's an example usage.

In [ ]:
dna = "TACAAAAATATT"

mrna = transcribe(dna)
protein = translate(mrna)

print("DNA:     ", dna)
print("mRNA:    ", mrna)
print("Protein: ", protein)

## Reverse DNA Transcription/Translation

Notice how each amino acid can be translated from multiple different codons. If we were given a certain protein, how could be reverse the process of transcription and translation to recover the original DNA template strand that created the protein? Let's go through this step by step.

First, we'll declare what we're given.

In [ ]:
# Given protein sequence
protein = ["Met", "Phe", "Stop"]

# Some example amino acids and what codons they translate from
aa_to_codons = {
    "Met":  ["AUG"],
    "Phe":  ["UUU", "UUC"],
    "Stop": ["UAA", "UAG", "UGA"],
}

Next, we'll set up our solver and its constraints.

In [ ]:
def solve_protein_to_dna(protein, reverse_codon_table, n=5):

    # Encoding each base as an integer for Z3
    A, U, G, C = 0, 1, 2, 3
    base_to_num = {"A": A, "U": U, "G": G, "C": C}
    num_to_base = {A: "A", U: "U", G: "G", C: "C"}

    s = Solver()

    # For each amino acid,
    # we want to define three Z3 variables: one for each mRNA base.
    # `mrna` holds these codon variables
    # mrna = [Int(f"mrna_{i}") for i in range(3 * len(protein))]
    mrna = [
        [Int(f"mrna_{i}_1"), Int(f"mrna_{i}_2"), Int(f"mrna_{i}_3")]
        for i in range(len(protein))
    ]

    # Each mRNA base must be A, U, G, or C
    for codon in mrna:
        for b in codon:
            s.add(Or(b == A, b == U, b == G, b == C))

    # Add codon constraints
    for i, aa in enumerate(protein):
        # Grab the ith codon, destructure into its three bases
        b1, b2, b3 = mrna[i]

        choices = []

        # Important part!!!
        for codon in reverse_codon_table[aa]:
            choices.append(
                And(
                    b1 == base_to_num[codon[0]],
                    b2 == base_to_num[codon[1]],
                    b3 == base_to_num[codon[2]],
                )
            )

        s.add(Or(choices))

    # Solve
    if s.check() == sat:
        print("Protein:     ", protein)
        print()

        # Print first n solutions that the solver found. n = 5 by default.
        n = 5
        i = 0
        while s.check() == sat and i < n:
            model = s.model()

            mrna_seq = ""

            for codon in mrna:
                mrna_seq += "".join(num_to_base[model[b].as_long()] for b in codon)

            # Reverse transcribe mRNA -> DNA template
            mrna_to_dna = {"A": "T", "U": "A", "G": "C", "C": "G"}

            dna_template = "".join(mrna_to_dna[b] for b in mrna_seq)

            print("mRNA:        ", mrna_seq)
            print("DNA template:", dna_template)
            print()

            # Block current solution
            s.add(Or([b != model[b] for codon in mrna for b in codon]))

            # Increment counter
            i += 1
    else:
        print("No solution")

solve_protein_to_dna(protein, aa_to_codons)

For fun, here's the same solver code above being used on a much larger codon table.

In [ ]:
# Given protein sequence
protein = ["Met", "Gly", "Ser", "Lys", "Stop"]

CODON_TABLE = {
    "UUU": "Phe", "UUC": "Phe", "UUA": "Leu", "UUG": "Leu",
    "UCU": "Ser", "UCC": "Ser", "UCA": "Ser", "UCG": "Ser",
    "UAU": "Tyr", "UAC": "Tyr", "UAA": "Stop", "UAG": "Stop",
    "UGU": "Cys", "UGC": "Cys", "UGA": "Stop", "UGG": "Trp",
    "CUU": "Leu", "CUC": "Leu", "CUA": "Leu", "CUG": "Leu",
    "CCU": "Pro", "CCC": "Pro", "CCA": "Pro", "CCG": "Pro",
    "CAU": "His", "CAC": "His", "CAA": "Gln", "CAG": "Gln",
    "CGU": "Arg", "CGC": "Arg", "CGA": "Arg", "CGG": "Arg",
    "AUU": "Ile", "AUC": "Ile", "AUA": "Ile", "AUG": "Met",
    "ACU": "Thr", "ACC": "Thr", "ACA": "Thr", "ACG": "Thr",
    "AAU": "Asn", "AAC": "Asn", "AAA": "Lys", "AAG": "Lys",
    "AGU": "Ser", "AGC": "Ser", "AGA": "Arg", "AGG": "Arg",
    "GUU": "Val", "GUC": "Val", "GUA": "Val", "GUG": "Val",
    "GCU": "Ala", "GCC": "Ala", "GCA": "Ala", "GCG": "Ala",
    "GAU": "Asp", "GAC": "Asp", "GAA": "Glu", "GAG": "Glu",
    "GGU": "Gly", "GGC": "Gly", "GGA": "Gly", "GGG": "Gly",
}

# Reverse mapping
from collections import defaultdict
REVERSE_CODON_TABLE = defaultdict(list)

for codon, aa in CODON_TABLE.items():
    REVERSE_CODON_TABLE[aa].append(codon)

# Solve it!
solve_protein_to_dna(protein, REVERSE_CODON_TABLE)


###Congratulations! You just used an SMT solver to recover the DNA behind a protein!


####If you'd like to continue your Z3 journey, you can start with this guide to learn more:
https://ericpony.github.io/z3py-tutorial/guide-examples.htm